# Day 11 — GANs & VAEs — Image Augmentation

## RetailMind AI Increment

### Objective
Generate synthetic product images for an under-represented fashion category using a GAN.

Workflow:

`Fashion Images → Find Rare Category → Train GAN → Generate Images → Inspect Quality → Augment Training Data → Re-evaluate Classifier`

## 1. Import Libraries

In [1]:
import os
import csv
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)

TensorFlow version: 2.21.0


## 2. Configuration

This notebook expects `../data/myntradataset/styles.csv` and `../data/myntradataset/images/` relative to the `notebooks/` folder.

In [ ]:
# This notebook is expected to be inside RETAILMIND_AI/notebooks/
DATASET_ROOT = Path('../data/myntradataset')
CSV_PATH = DATASET_ROOT / 'styles.csv'
IMAGES_DIR = DATASET_ROOT / 'images'

if not CSV_PATH.exists():
    raise FileNotFoundError(f'CSV file not found: {CSV_PATH.resolve()}')
if not IMAGES_DIR.is_dir():
    raise FileNotFoundError(f'Images directory not found: {IMAGES_DIR.resolve()}')

OUTPUT_DIR = Path('../data/day11_outputs')
GENERATED_DIR = OUTPUT_DIR / 'generated_images'
SAMPLE_DIR = OUTPUT_DIR / 'samples'

OUTPUT_DIR.mkdir(exist_ok=True)
GENERATED_DIR.mkdir(exist_ok=True)
SAMPLE_DIR.mkdir(exist_ok=True)

IMAGE_SIZE = 64
BATCH_SIZE = 4  # Lower batch size to reduce RAM usage
LATENT_DIM = 128
EPOCHS = 2  # Start small; increase only after a successful run

print('Dataset:', DATASET_ROOT)

Dataset: ..\data\myntradataset


## 3. Read Fashion Product Metadata

In [3]:
import csv

records = []

with open(CSV_PATH, 'r', encoding='utf-8', errors='ignore') as file:
    reader = csv.DictReader(file)

    for row in reader:
        records.append(row)

print("Total records:", len(records))
print(records[:2])

Total records: 44446
[{'id': '15970', 'gender': 'Men', 'masterCategory': 'Apparel', 'subCategory': 'Topwear', 'articleType': 'Shirts', 'baseColour': 'Navy Blue', 'season': 'Fall', 'year': '2011', 'usage': 'Casual', 'productDisplayName': 'Turtle Check Men Navy Blue Shirt'}, {'id': '39386', 'gender': 'Men', 'masterCategory': 'Apparel', 'subCategory': 'Bottomwear', 'articleType': 'Jeans', 'baseColour': 'Blue', 'season': 'Summer', 'year': '2012', 'usage': 'Casual', 'productDisplayName': 'Peter England Men Party Blue Jeans'}]


## 4. Count Product Categories

This helps identify under-represented categories.

In [4]:
category_counts = {}

for item in records:
    category = item['subCategory']
    category_counts[category] = category_counts.get(category, 0) + 1

sorted_categories = sorted(
    category_counts.items(),
    key=lambda x: x[1]
)

print(sorted_categories)

[('Home Furnishing', 1), ('Vouchers', 1), ('Sports Accessories', 3), ('Beauty Accessories', 4), ('Wristbands', 4), ('Perfumes', 6), ('Umbrellas', 6), ('Water Bottle', 7), ('Bath and Body', 12), ('Hair', 19), ('Gloves', 20), ('Sports Equipment', 21), ('Shoe Accessories', 24), ('Mufflers', 38), ('Eyes', 43), ('Skin', 69), ('Skin Care', 77), ('Stoles', 90), ('Free Gifts', 104), ('Apparel Set', 106), ('Cufflinks', 108), ('Scarves', 118), ('Accessories', 143), ('Ties', 258), ('Headwear', 293), ('Makeup', 307), ('Nails', 329), ('Saree', 427), ('Loungewear and Nightwear', 470), ('Dress', 478), ('Lips', 527), ('Socks', 698), ('Belts', 811), ('Flip Flops', 915), ('Wallets', 933), ('Sandal', 963), ('Fragrance', 1012), ('Eyewear', 1073), ('Jewellery', 1080), ('Innerwear', 1808), ('Watches', 2542), ('Bottomwear', 2694), ('Bags', 3055), ('Shoes', 7344), ('Topwear', 15405)]


## 5. Select Under-Represented Category

Ideally, set this manually using the weakest class from your previous Day 6/7 confusion matrix.

If `TARGET_CATEGORY` is `None`, the notebook automatically selects the smallest category.

In [5]:
# Set this to a valid subCategory from the printed category list.
TARGET_CATEGORY = 'Topwear'

if TARGET_CATEGORY not in category_counts:
    raise ValueError(
        f'{TARGET_CATEGORY!r} is not a subCategory. '
        'Choose one from the category counts printed above.'
    )

print('Selected category:', TARGET_CATEGORY)
print('Number of real images:', category_counts[TARGET_CATEGORY])

Selected category: Topwear
Number of real images: 15405


## 6. Collect Target Category Images

In [6]:
# Keep only image files that actually exist on disk.
target_images = []
missing_images = []

for item in records:
    if item['subCategory'] == TARGET_CATEGORY:
        image_path = IMAGES_DIR / f"{item['id']}.jpg"
        if image_path.is_file():
            target_images.append(str(image_path))
        else:
            missing_images.append(str(image_path))

print('Target category:', TARGET_CATEGORY)
print('Valid training images:', len(target_images))
print('Missing image files:', len(missing_images))

if len(target_images) < BATCH_SIZE:
    raise ValueError(
        'Not enough valid images for this category. '
        'Choose a more common subCategory or reduce BATCH_SIZE.'
    )

Target category: Topwear
Valid training images: 3618
Missing image files: 11787


## 7. Image Preprocessing

GAN images are resized to `64 × 64 × 3` and normalized to `[-1, 1]`.

In [7]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    image = tf.cast(image, tf.float32)

    image = (image - 127.5) / 127.5

    return image

## 8. Create TensorFlow Dataset

In [8]:
dataset = tf.data.Dataset.from_tensor_slices(target_images)

# Smaller buffers and limited parallelism reduce RAM pressure.
dataset = dataset.shuffle(min(len(target_images), 1000))
dataset = dataset.map(load_image, num_parallel_calls=1)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)
dataset = dataset.prefetch(1)

print(dataset)

<_PrefetchDataset element_spec=TensorSpec(shape=(4, 64, 64, 3), dtype=tf.float32, name=None)>


## 9. Check Image Shape

In [9]:
sample_batch = next(iter(dataset))

print('Batch shape:', sample_batch.shape)
print('Minimum value:', tf.reduce_min(sample_batch).numpy())
print('Maximum value:', tf.reduce_max(sample_batch).numpy())

Batch shape: (4, 64, 64, 3)
Minimum value: -1.0
Maximum value: 1.0


# GAN Architecture

A GAN contains two neural networks:

```text
Generator
    ↓
Creates fake images

Discriminator
    ↓
Determines real vs fake
```

The two models compete during training.

## 10. Build Generator

In [10]:
def build_generator():

    model = keras.Sequential([

        layers.Input(shape=(LATENT_DIM,)),

        layers.Dense(8 * 8 * 256, use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(negative_slope=0.2),

        layers.Reshape((8, 8, 256)),

        layers.Conv2DTranspose(
            128,
            kernel_size=4,
            strides=2,
            padding='same',
            use_bias=False
        ),
        layers.BatchNormalization(),
        layers.LeakyReLU(negative_slope=0.2),

        layers.Conv2DTranspose(
            64,
            kernel_size=4,
            strides=2,
            padding='same',
            use_bias=False
        ),
        layers.BatchNormalization(),
        layers.LeakyReLU(negative_slope=0.2),

        layers.Conv2DTranspose(
            3,
            kernel_size=4,
            strides=2,
            padding='same',
            activation='tanh'
        )
    ])

    return model

In [11]:
generator = build_generator()

generator.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16384)          │     2,097,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 16384)          │        65,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose                │ (None, 16, 16, 128)    │       524,288 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_1              │ (None, 32, 32, 64)     │       131,072 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_2              │ (None, 64, 64, 3)      │         3,075 │
│ (Conv2DTranspose)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,821,891 (10.76 MB)

 Trainable params: 2,788,739 (10.64 MB)

 Non-trainable params: 33,152 (129.50 KB)

## 11. Test Generator

In [12]:
noise = tf.random.normal([1, LATENT_DIM])

generated_image = generator(noise, training=False)

print('Generated image shape:', generated_image.shape)

Generated image shape: (1, 64, 64, 3)


## 12. Build Discriminator

In [13]:
def build_discriminator():

    model = keras.Sequential([

        layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),

        layers.Conv2D(
            64,
            kernel_size=4,
            strides=2,
            padding='same'
        ),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dropout(0.3),

        layers.Conv2D(
            128,
            kernel_size=4,
            strides=2,
            padding='same'
        ),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dropout(0.3),

        layers.Conv2D(
            256,
            kernel_size=4,
            strides=2,
            padding='same'
        ),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1)
    ])

    return model

In [14]:
discriminator = build_discriminator()

discriminator.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 64)     │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 128)    │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 256)      │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_5 (LeakyReLU)       │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │        16,385 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 675,265 (2.58 MB)

 Trainable params: 675,265 (2.58 MB)

 Non-trainable params: 0 (0.00 B)

## 13. Define GAN Loss Functions

In [15]:
cross_entropy = keras.losses.BinaryCrossentropy(
    from_logits=True
)


def discriminator_loss(real_output, fake_output):

    real_loss = cross_entropy(
        tf.ones_like(real_output),
        real_output
    )

    fake_loss = cross_entropy(
        tf.zeros_like(fake_output),
        fake_output
    )

    return real_loss + fake_loss


def generator_loss(fake_output):

    return cross_entropy(
        tf.ones_like(fake_output),
        fake_output
    )

## 14. Create Optimizers

In [16]:
generator_optimizer = keras.optimizers.Adam(
    learning_rate=0.0002,
    beta_1=0.5
)

discriminator_optimizer = keras.optimizers.Adam(
    learning_rate=0.0002,
    beta_1=0.5
)

## 15. GAN Training Step

The discriminator learns to distinguish real images from generated images.

The generator learns to fool the discriminator.

In [17]:
@tf.function
def train_step(real_images):

    batch_size = tf.shape(real_images)[0]

    noise = tf.random.normal([
        batch_size,
        LATENT_DIM
    ])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:

        generated_images = generator(
            noise,
            training=True
        )

        real_output = discriminator(
            real_images,
            training=True
        )

        fake_output = discriminator(
            generated_images,
            training=True
        )

        gen_loss = generator_loss(fake_output)

        disc_loss = discriminator_loss(
            real_output,
            fake_output
        )

    generator_gradients = gen_tape.gradient(
        gen_loss,
        generator.trainable_variables
    )

    discriminator_gradients = disc_tape.gradient(
        disc_loss,
        discriminator.trainable_variables
    )

    generator_optimizer.apply_gradients(
        zip(
            generator_gradients,
            generator.trainable_variables
        )
    )

    discriminator_optimizer.apply_gradients(
        zip(
            discriminator_gradients,
            discriminator.trainable_variables
        )
    )

    return gen_loss, disc_loss

## 16. Create Fixed Noise

Using the same noise after each epoch lets us compare how generated images improve during training.

In [18]:
NUM_EXAMPLES = 16

fixed_noise = tf.random.normal([
    NUM_EXAMPLES,
    LATENT_DIM
])

## 17. Save Generated Samples

In [19]:
def save_generated_samples(epoch):

    predictions = generator(
        fixed_noise,
        training=False
    )

    predictions = (predictions + 1.0) * 127.5
    predictions = tf.clip_by_value(predictions, 0, 255)
    predictions = tf.cast(predictions, tf.uint8)

    for i, image in enumerate(predictions):

        path = SAMPLE_DIR / f'epoch_{epoch}_image_{i}.png'

        encoded = tf.io.encode_png(image)
        tf.io.write_file(str(path), encoded)

## 18. Train GAN

GAN training can be unstable. Examine generator and discriminator losses together with generated images.

In [20]:
generator_losses = []
discriminator_losses = []

for epoch in range(1, EPOCHS + 1):

    gen_metric = keras.metrics.Mean()
    disc_metric = keras.metrics.Mean()

    for image_batch in dataset:

        gen_loss, disc_loss = train_step(image_batch)

        gen_metric.update_state(gen_loss)
        disc_metric.update_state(disc_loss)

    avg_gen_loss = float(gen_metric.result())
    avg_disc_loss = float(disc_metric.result())

    generator_losses.append(avg_gen_loss)
    discriminator_losses.append(avg_disc_loss)

    print(
        f'Epoch {epoch}/{EPOCHS} | '
        f'Generator Loss: {avg_gen_loss:.4f} | '
        f'Discriminator Loss: {avg_disc_loss:.4f}'
    )

    if epoch == 1 or epoch % 5 == 0:
        save_generated_samples(epoch)

Epoch 1/2 | Generator Loss: 1.1540 | Discriminator Loss: 1.0342
Epoch 2/2 | Generator Loss: 0.9409 | Discriminator Loss: 1.1927


# Mode Collapse Inspection

Mode collapse happens when the generator repeatedly produces very similar images instead of diverse samples.

We calculate a simple diversity score and also manually inspect generated images.

Low diversity can indicate mode collapse.

## 19. Generate Images for Diversity Check

In [21]:
quality_noise = tf.random.normal([
    32,
    LATENT_DIM
])

quality_images = generator(
    quality_noise,
    training=False
)

print('Generated batch:', quality_images.shape)

Generated batch: (32, 64, 64, 3)


## 20. Simple Diversity Score

In [22]:
def diversity_score(images):

    flattened = tf.reshape(
        images,
        [tf.shape(images)[0], -1]
    )

    differences = (
        tf.expand_dims(flattened, 1)
        - tf.expand_dims(flattened, 0)
    )

    distances = tf.sqrt(
        tf.reduce_mean(
            tf.square(differences),
            axis=2
        ) + 1e-8
    )

    count = tf.cast(
        tf.shape(flattened)[0],
        tf.float32
    )

    score = tf.reduce_sum(distances) / (
        count * (count - 1)
    )

    return float(score)


score = diversity_score(quality_images)

print('Diversity score:', round(score, 4))

Diversity score: 0.6202


## 21. Quality Checklist

Inspect the images stored inside `day11_outputs/samples/`.

Keep generated images only when they have:

- recognizable product shape;
- plausible colors;
- reasonable product structure;
- sufficient diversity;
- no severe artifacts; and
- no obvious mode collapse.

The diversity score is only a warning signal. Visual inspection is still required.

# Generate Synthetic Training Images

## 22. Generate Synthetic Images

In [23]:
NUM_SYNTHETIC_IMAGES = 100

noise = tf.random.normal([
    NUM_SYNTHETIC_IMAGES,
    LATENT_DIM
])

synthetic_images = generator(
    noise,
    training=False
)

print('Synthetic images:', synthetic_images.shape)

Synthetic images: (100, 64, 64, 3)


## 23. Save Synthetic Images

In [24]:
category_name = TARGET_CATEGORY.replace(' ', '_')

category_output_dir = GENERATED_DIR / category_name
category_output_dir.mkdir(exist_ok=True)

synthetic_images_uint8 = (
    synthetic_images + 1.0
) * 127.5

synthetic_images_uint8 = tf.cast(
    tf.clip_by_value(
        synthetic_images_uint8,
        0,
        255
    ),
    tf.uint8
)

for i, image in enumerate(synthetic_images_uint8):

    output_path = (
        category_output_dir
        / f'synthetic_{i:04d}.png'
    )

    tf.io.write_file(
        str(output_path),
        tf.io.encode_png(image)
    )

print('Saved:', NUM_SYNTHETIC_IMAGES)
print('Location:', category_output_dir)

Saved: 100
Location: day11_outputs\generated_images\Topwear


# Model Persistence

## 24. Save Generator Model

The generator can later be reused to create additional synthetic images.

In [25]:
generator.save(
    OUTPUT_DIR / 'fashion_generator.keras'
)

print('Generator saved.')

Generator saved.


## 25. Load Saved Generator

In [26]:
loaded_generator = keras.models.load_model(
    OUTPUT_DIR / 'fashion_generator.keras'
)

print('Generator loaded successfully.')

Generator loaded successfully.


# Fold Synthetic Images Into Existing Classifier

After inspecting and approving realistic images, add them only to the **training set** of the under-represented category.

Do not add generated images to validation or test data.

The comparison must keep the original real test set unchanged.

## 26. Before/After Evaluation Template

Run the same Day 6/7 classifier before and after augmentation.

Record the metrics below from the unchanged test set.

In [27]:
results = {
    'target_category': TARGET_CATEGORY,
    'real_images': len(target_images),
    'missing_images': len(missing_images),
    'synthetic_images_generated': NUM_SYNTHETIC_IMAGES,
    'target_class_accuracy_before': None,
    'target_class_accuracy_after': None,
    'overall_accuracy_before': None,
    'overall_accuracy_after': None
}

for key, value in results.items():
    print(key, ':', value)

target_category : Topwear
real_images : 3618
missing_images : 11787
synthetic_images_generated : 100
target_class_accuracy_before : None
target_class_accuracy_after : None
overall_accuracy_before : None
overall_accuracy_after : None


## 27. Calculate Improvement

Enter the real results from the Day 6/7 classifier first.

In [28]:
before = results['target_class_accuracy_before']
after = results['target_class_accuracy_after']

if before is not None and after is not None:
    improvement = after - before
    print('Target class improvement:', improvement)
else:
    print('Enter the before/after classifier metrics first.')

Enter the before/after classifier metrics first.


# Day 11 Final Validation

In [29]:
assert generator.output_shape == (None, 64, 64, 3)

assert discriminator.output_shape == (None, 1)

assert synthetic_images.shape[0] == NUM_SYNTHETIC_IMAGES

assert category_output_dir.exists()

assert (OUTPUT_DIR / 'fashion_generator.keras').exists()

print('All Day 11 validation checks passed.')

All Day 11 validation checks passed.


# Day 11 Summary

In this RetailMind increment we implemented:

- Fashion Product Images loading
- Product-category counting
- Under-represented category selection
- TensorFlow image preprocessing
- DCGAN Generator
- DCGAN Discriminator
- GAN loss functions
- Custom GAN training with GradientTape
- Generated sample inspection
- Basic mode-collapse/diversity checking
- Synthetic product-image generation
- Generator persistence
- Training-set augmentation preparation
- Before/after classifier evaluation template

The final fairness/accuracy conclusion should be based on the existing Day 6/7 classifier evaluated on the same unchanged real test set before and after synthetic augmentation.